# 05 - GRPO with a local drug-target verifier

This notebook uses a local copy of `drug_bank.csv` supplied by the user at runtime. The source data is not stored in this public repository.

Pipeline: local drug-target table -> compact dictionary -> prompt-only GRPO -> verifiable reward -> drug-disjoint evaluation -> before/after comparison.

**Important:** use the source data only according to its original terms.


In [ ]:
!pip -q install -U \
    "transformers>=4.55,<5" \
    "datasets>=3.6,<5" \
    "peft>=0.17,<1" \
    "trl>=0.29,<1" \
    "accelerate>=1.10,<2" \
    "bitsandbytes>=0.46,<1" \
    "torchao>=0.16,<1"


In [ ]:
import ast
import re
import pandas as pd
import torch
from datasets import Dataset

import transformers, datasets, peft, trl
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before running this notebook.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


## 1. Upload `drug_bank.csv`

Upload the local source file directly into the Colab runtime. Nothing is fetched from the GitHub repository.


In [ ]:
from google.colab import files

uploaded = files.upload()
if "drug_bank.csv" not in uploaded:
    raise FileNotFoundError("Please upload a file named drug_bank.csv")
DATA_PATH = "/content/drug_bank.csv"
print("Using:", DATA_PATH)


In [ ]:
df = pd.read_csv(DATA_PATH)
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()


## 2. Build a compact drug -> target dictionary

For the first GRPO experiment, keep only drugs with exactly one target in the source table. The gold target is the UniProt ID.


In [ ]:
drug_targets = {}
for _, row in df.iterrows():
    drug = str(row["Drug Name"]).strip()
    try:
        target_ids = ast.literal_eval(str(row["Targets"]))
    except Exception:
        target_ids = []
    try:
        target_names = ast.literal_eval(str(row["Targets Name"]))
    except Exception:
        target_names = []
    pairs = list(zip(target_ids, target_names))
    if drug and pairs:
        drug_targets[drug] = pairs

single_target = {drug: pairs[0] for drug, pairs in drug_targets.items() if len(pairs) == 1}
print("Drugs with exactly one target:", len(single_target))
print(list(single_target.items())[:10])


## 3. Build a prompt-only GRPO dataset

GRPO receives prompts. The reward function can access extra dataset columns such as the gold target. We split by drug so evaluation uses unseen drugs.


In [ ]:
records = []
for drug, (target_id, target_name) in single_target.items():
    records.append({
        "prompt": [{
            "role": "user",
            "content": f"What is the protein target UniProt ID for the drug {drug}? Reply with the UniProt ID only."
        }],
        "drug": drug,
        "target_id": target_id,
        "target_name": target_name,
    })

dataset = Dataset.from_list(records).shuffle(seed=42)
split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print("Total:", len(dataset))
print("Train:", len(train_dataset))
print("Eval:", len(eval_dataset))
print(train_dataset[0])


## 4. Load Qwen2.5-0.5B-Instruct


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).cuda()
print("Has chat template:", tokenizer.chat_template is not None)


## 5. Generation and exact-match evaluation


In [ ]:
def make_inputs(drug):
    messages = [{
        "role": "user",
        "content": f"What is the protein target UniProt ID for the drug {drug}? Reply with the UniProt ID only."
    }]
    return tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True)

def generate_answer(model, drug, max_new_tokens=16, do_sample=False):
    encoded = make_inputs(drug)
    encoded = {key: value.to(model.device) for key, value in encoded.items()}
    with torch.no_grad():
        output = model.generate(**encoded, max_new_tokens=max_new_tokens, do_sample=do_sample, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(output[0], skip_special_tokens=True)

def extract_target(text):
    found = re.findall(r"\b[A-Z][0-9][A-Z0-9]{3}[0-9]\b", text.upper())
    return found[0] if found else None

def exact_accuracy(model, dataset):
    rows = []
    correct = 0
    for example in dataset:
        text = generate_answer(model, example["drug"])
        pred = extract_target(text)
        gold = str(example["target_id"]).upper()
        ok = pred == gold
        correct += int(ok)
        rows.append({"drug": example["drug"], "gold": gold, "pred": pred, "correct": ok})
    accuracy = correct / len(dataset) if len(dataset) else float("nan")
    return accuracy, rows

before_acc, before_rows = exact_accuracy(model, eval_dataset)
print(f"Before GRPO exact target accuracy: {before_acc:.3f}")
for row in before_rows[:10]:
    print(row)


## 6. Verifiable reward

The reward is 1 when the generated answer contains the gold UniProt ID, otherwise 0. This is intentionally simple and fully deterministic.


In [ ]:
def target_reward(completions, target_id, **kwargs):
    rewards = []
    for completion, gold in zip(completions, target_id):
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        found = re.findall(r"\b[A-Z][0-9][A-Z0-9]{3}[0-9]\b", text.upper())
        rewards.append(1.0 if str(gold).upper() in found else 0.0)
    return rewards


## 7. GRPO + LoRA

GRPO samples multiple candidate completions and uses the verifier reward to update the policy.


In [ ]:
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

grp_args = GRPOConfig(
    output_dir="./outputs/drug-target-grpo",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_steps=100,
    num_generations=4,
    max_prompt_length=128,
    max_completion_length=16,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    use_vllm=False,
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
)

trainer = GRPOTrainer(
    model=model,
    args=grp_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    reward_funcs=target_reward,
    peft_config=lora_config,
)
trainer.train()


## 8. After-GRPO evaluation


In [ ]:
after_acc, after_rows = exact_accuracy(model, eval_dataset)
print(f"After GRPO exact target accuracy: {after_acc:.3f}")
for row in after_rows[:10]:
    print(row)


## 9. Before vs After

The same drug-disjoint evaluation set is used for both measurements.


In [ ]:
print("Metric comparison")
print("-" * 70)
print(f"{'Metric':<30}{'Before':>12}{'After':>12}{'Delta':>12}")
print("-" * 70)
print(f"{'Exact target accuracy':<30}{before_acc:>12.3f}{after_acc:>12.3f}{after_acc-before_acc:>+12.3f}")
print("-" * 70)


## 10. Inspect a few predictions

Read these outputs critically. Exact matching measures retrieval against the source table; it does not establish clinical correctness.


In [ ]:
for before, after in zip(before_rows[:5], after_rows[:5]):
    print("=" * 80)
    print("DRUG:", before["drug"])
    print("GOLD:", before["gold"])
    print("BEFORE PRED:", before["pred"])
    print("AFTER PRED :", after["pred"])


## Next experiments

1. Use all valid targets per drug instead of restricting to single-target drugs.
2. Add a format reward plus a target reward.
3. Replace exact-match-only reward with partial credit or hard-negative-aware reward.
4. Compare GRPO against SFT and DPO on the same drug-disjoint benchmark.
